# 資料を調べて答えるチャットボットを作る (複数ターンの RAG)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_dialogue/chat_rag.ipynb)

NLP クックブック 19-08 ／ 対話システム

対話の流れを踏まえながら、手元の資料を検索して答えるチャットボットを作りたい。雑談と資料に基づく回答を両立させたい。

In [1]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

docs = [
    "クックブック食堂の営業時間は平日 11 時から 15 時、土日は 11 時から 20 時です。月曜日は定休日です。",
    "クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレー (950 円) です。大盛りは 100 円増しです。",
    "クックブック食堂は 2019 年に京都市左京区で開業しました。店主の田中さんはインドで 2 年間修行しました。",
    "駐車場は店舗の裏に 3 台分あります。満車の場合は近隣のコインパーキングをご利用ください。",
    "予約は電話 (075-000-0000) のみ受け付けています。5 名以上の場合は前日までにご連絡ください。",
    "支払いは現金と QR コード決済に対応しています。クレジットカードは使えません。",
    "辛さは 5 段階から選べます。お子様向けの甘口カレー (600 円) もあります。",
]
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
doc_emb = embedder.encode(["passage: " + d for d in docs], normalize_embeddings=True)

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def generate(messages, max_new_tokens=80):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [2]:
SYSTEM = ("あなたはクックブック食堂の案内係です。丁寧な言葉で、2 文以内で答えてください。"
          "お店に関する質問には、与えられた資料に書かれていることだけを使って答え、資料にないことは「わかりかねます」と答えてください。")

class RagChat:
    def __init__(self, threshold=0.80):
        self.history, self.threshold = [], threshold

    def rewrite(self, question):
        """対話の文脈を補って、検索用の質問にする (最初のターンはそのまま)"""
        if not self.history:
            return question
        dialogue = "\n".join(f"{'ユーザー' if m['role'] == 'user' else 'アシスタント'}: {m['content']}" for m in self.history[-4:])
        prompt = ("次の対話の最後の質問を、対話を読んでいない人にも意味が通じる、検索用の 1 つの質問に書き換えてください。"
                  f"書き換えた質問だけを出力してください。\n\n{dialogue}\nユーザー: {question}")
        return generate([{"role": "user", "content": prompt}], max_new_tokens=50).strip("「」")

    def say(self, question):
        query = self.rewrite(question)
        sims = doc_emb @ embedder.encode("query: " + query, normalize_embeddings=True)
        hits = [int(i) for i in sims.argsort()[::-1][:2] if sims[i] >= self.threshold]
        # 資料は履歴に残さず、今回の質問にだけ付ける (履歴が資料で膨らむのを防ぐ)
        content = question if not hits else "# 資料\n" + "\n".join(f"- {docs[i]}" for i in hits) + f"\n\n# 質問\n{question}"
        reply = generate([{"role": "system", "content": SYSTEM}] + self.history[-6:] + [{"role": "user", "content": content}])
        self.history += [{"role": "user", "content": question}, {"role": "assistant", "content": reply}]
        return reply, query, hits

bot = RagChat()
for user in ["こんにちは。", "おすすめのメニューは何ですか？", "それはいくらですか？", "辛いのが苦手なのですが大丈夫でしょうか。",
             "日曜の夜に行きたいのですが、開いていますか？", "カードは使えますか？", "ペットを連れて行けますか？"]:
    reply, query, hits = bot.say(user)
    print(f"あなた: {user}\nボット: {reply}\n   (検索クエリ: {query} / 参照した資料: {hits})")

あなた: こんにちは。
ボット: こんにちは。当店では予約を電話でのみ受け付けており、電話番号は 075-000-0000 です。5 名様以上でのご利用の場合は、前日Untilまでにご連絡いただけるとスムーズです。また、辛さは 5 段階からお選びいただけるほか、お子様向けの甘口カレー (600 円) もご用意しております。
   (検索クエリ: こんにちは。 / 参照した資料: [4, 6])
あなた: おすすめのメニューは何ですか？
ボット: 当店おすすめのメニューは、3 日間煮込んだ特製ビーフカレー (950 円) です。お時間許す限りじっくりと煮込んだ、深みのある味わいをお楽しみください。大盛り (1050 円) にも変更できます。
   (検索クエリ: この会話の中で、特に強調されていたサービスやメニューは何ですか？ / 参照した資料: [1])
あなた: それはいくらですか？
ボット: 特製ビーフカレーの大盛りは、通常価格より 100 円増しの 1050 円でご提供しております。ただし、ご質問いただいたのが通常のビーフカレーについてであれば、価格は 950 円です。資料では大盛り価格のみ記載していたため、通常の価格についてもお答えしました。もし通常のビーフカレーの価格のみをお知りになりたい場合は、「通常のビーフカレーは 950
   (検索クエリ: 3 日間煮込んだ特製ビーフカレーの値段と、その大盛り料金について教えてください。 / 参照した資料: [1, 6])
あなた: 辛いのが苦手なのですが大丈夫でしょうか。
ボット: もちろんです。辛いのが苦手なお客様には、甘口カレー (600 円) をご用意しておりますので、そちらをご利用ください。また、通常のビーフカレーや他のメニューも、辛さを調節することが可能ですので、ご注文時にお申し付けください。お客様のお好みに合わせた辛さでご提供いたします。
   (検索クエリ: 特製ビーフカレーの辛さレベルと、辛いのが苦手な方向けの調整が可能かどうか教えてください。 / 参照した資料: [6, 1])
あなた: 日曜の夜に行きたいのですが、開いていますか？
ボット: 申し訳ございません。クックブック食堂は土日も営業しておりますが、営業時間は 20 時までとなっておりますため、日曜の夜には営業しておりません。翌営業日の平

- 各ターンで「質問の書き換え → 検索 → 資料を付けて生成」を行う。「それはいくら」のような質問は、書き換えないと検索に当たらない。
- 検索した資料は、そのターンの入力にだけ付けて、履歴には質問と回答だけを残す。資料まで履歴に積むと、数ターンで入力があふれる。
- 類似度が低いときは資料を付けない。挨拶や雑談に無関係な資料を付けると、それに引きずられた応答になる。
- 資料にないこと (ペット) を聞かれたときに正直に断れるかは、モデルの大きさとプロンプトに左右される。RAG の章の「わかりませんと答えさせる」「回答を評価する」と合わせて確認する。